# Building the final sample

This notebook applies every cut used in the midplane analysis to the SDSS-V Astra ASPCAP catalog and saves the remaining stars to a file, so later notebooks can load it instead of repeating the cuts.

**Inputs**
- `astraAllStarASPCAP-0.8.fits.gz`: SDSS-V Astra ASPCAP catalog
- `allStarLite-dr17-synspec_rev1.fits`: SDSS-IV DR17 catalog, used only as a fallback source of S/N

**Output**
- `final_sample_zsun0.fits`: every original catalog column for each star that passes all the cuts, plus computed columns (distance, Galactocentric x, y, z, R, phi, [Mg/Fe] and its uncertainty, and the S/N we used)
- `final_sample_cut_log.csv`: how many stars remain after each cut

**How to read it:** each section below is one stage of cuts, with a short description, the code, and the number of stars left. The chemistry and coordinate cuts use the functions in `src/` (`chemistry.py`, `coords.py`), so they are exactly the cuts used in the analysis. The quality cuts before the chemistry are written out step by step here. We do not cut on radial velocity or proper motion.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import astropy.table as at
import astropy.units as u
import astropy.coordinates as coord
from astropy.table import Table
from astropy.coordinates import SkyCoord, Distance

# --- make the midplane package importable ---------------------------------
# This notebook lives in  <repo>/notebooks/  and the code is in  <repo>/src/midplane/
REPO_ROOT = Path.cwd().parent            # folder above notebooks/
sys.path.insert(0, str(REPO_ROOT / "src"))

# helpers and cut definitions from src/midplane/
from midplane.utils import col_to_float_array, masked_fallback_array
from midplane.selection import build_gb_polygon, giant_branch_mask
from midplane.config import SNR_MIN

# --- input and output files -----------------------------------------------
DATA_DIR   = REPO_ROOT / "data"          # CHANGE to the folder holding the two catalog files
ASTRA_FILE = DATA_DIR / "astraAllStarASPCAP-0.8.fits.gz"
DR17_FILE  = DATA_DIR / "allStarLite-dr17-synspec_rev1.fits"
OUT_SAMPLE = DATA_DIR / "final_sample_zsun0.fits"
OUT_LOG    = DATA_DIR / "final_sample_cut_log.csv"

# --- cut values that are not defined in src/ ------------------------------
FEH_MIN_EXTRA = -0.9     # extra [Fe/H] floor, applied after the src [Fe/H] cut
R_MAX_KPC     = 16.0     # keep stars with Galactocentric R below this

# --- running log of (step, number of stars left) --------------------------
cut_log = []

## 1. Load the catalogs, join DR17, and keep one row per star

The parent sample is the Astra ASPCAP catalog. We left-join the DR17 catalog onto it with the SDSS-IV APOGEE ID, which adds two DR17 columns, `SNR` and `VHELIO_AVG`. Stars with no DR17 counterpart (for example stars first observed in SDSS-V) stay in the table with those columns masked. DR17 is only a fallback for S/N: we use the Astra `snr` where it exists and the DR17 `SNR` where it doesn't.

Both the Astra file and the join can produce several rows for the same star. We keep one row per `sdss_id`. Among the copies we prefer a clean one (not flagged bad, no abundance flags, finite [Fe/H], [Mg/H] and S/N) and, within that, the one with the highest S/N. Rows with no valid `sdss_id` are all kept.

The checks at the end count how many stars are crossmatched to a DR17 star with a S/N, and where the S/N used by the cuts comes from.

In [4]:
# --- Astra ASPCAP catalog (the parent sample) ---
t = Table.read(ASTRA_FILE, format="fits", hdu=2)
t["sdss4_apogee_id"] = t["sdss4_apogee_id"].filled(-1)   # no SDSS-IV counterpart -> -1
n_astra = len(t)
cut_log.append(("0a. Astra ASPCAP rows in the file", n_astra))

# --- DR17 allStarLite: only the columns we need ---
dr17 = Table.read(DR17_FILE, hdu=1)["APOGEE_ID", "SNR", "VHELIO_AVG"]

# --- left join on the SDSS-IV APOGEE ID (adds rows when DR17 lists a star twice) ---
t = at.join(
    t, dr17,
    keys_left="sdss4_apogee_id", keys_right="APOGEE_ID",
    join_type="left",
)
n_joined = len(t)
cut_log.append(("0b. after joining DR17 (includes duplicate rows)", n_joined))

# --- keep one row per sdss_id: the best usable copy ---
sid = np.ma.filled(t["sdss_id"], -1).astype(np.int64)
valid = sid >= 0

snr_use = masked_fallback_array(t, "snr", "SNR")             # same S/N the cuts use
snr_key = np.where(np.isfinite(snr_use), snr_use, -np.inf)

# a "clean" copy: not flagged bad, no abundance flags, finite [Fe/H], [Mg/H] and S/N
clean = (~np.asarray(t["flag_bad"], bool)
         & (np.asarray(t["fe_h_flags"]) == 0) & (np.asarray(t["mg_h_flags"]) == 0)
         & np.isfinite(col_to_float_array(t, "fe_h"))
         & np.isfinite(col_to_float_array(t, "mg_h"))
         & np.isfinite(snr_use))

# sort by sdss_id, then clean copies first, then highest S/N first
order = np.lexsort((-snr_key, ~clean, sid))
sid_sorted = sid[order]
is_first = np.r_[True, sid_sorted[1:] != sid_sorted[:-1]]
keep = np.sort(order[is_first | ~valid[order]])              # all invalid ids are kept
t = t[keep]
n_parent = len(t)
cut_log.append(("0c. one row per sdss_id (parent catalog)", n_parent))

print(f"Astra rows in the file:                {n_astra:>10,d}")
print(f"After joining DR17:                    {n_joined:>10,d}   ({n_joined - n_astra:+,d})")
print(f"After one row per sdss_id (parent):    {n_parent:>10,d}   ({n_parent - n_joined:+,d})")
print(f"  parent rows that are clean copies:   {int(clean[keep].sum()):>10,d}")

# =============================================================================
# CHECK: how many Astra stars are crossmatched to a DR17 star with a S/N?
# =============================================================================
ids = np.asarray(t["sdss4_apogee_id"]).astype(str)
has_sdss4_id = ~np.isin(ids, ["-1", ""])

snr_dr17  = col_to_float_array(t, "SNR")
snr_astra = col_to_float_array(t, "snr")
dr17_snr_ok  = np.isfinite(snr_dr17)
astra_snr_ok = np.isfinite(snr_astra)

print()
print(f"Stars in the parent catalog:                       {n_parent:>10,d}")
print(f"  with an SDSS-IV APOGEE ID:                       {has_sdss4_id.sum():>10,d}")
print(f"  crossmatched to a DR17 star with a S/N:          {dr17_snr_ok.sum():>10,d}"
      f"  ({100 * dr17_snr_ok.sum() / n_parent:.1f}%)")
print(f"  with no DR17 match:                              {(~dr17_snr_ok).sum():>10,d}")
print()
print("Where the S/N we use comes from:")
print(f"  Astra S/N available:                             {astra_snr_ok.sum():>10,d}")
print(f"  Astra S/N missing, DR17 S/N available (fallback):{(~astra_snr_ok & dr17_snr_ok).sum():>10,d}")
print(f"  no S/N from either source:                       {(~astra_snr_ok & ~dr17_snr_ok).sum():>10,d}")

Astra rows in the file:                 2,466,006
After joining DR17:                     2,576,187   (+110,181)
After one row per sdss_id (parent):     2,272,875   (-303,312)
  parent rows that are clean copies:    1,754,276

Stars in the parent catalog:                        2,272,875
  with an SDSS-IV APOGEE ID:                          457,425
  crossmatched to a DR17 star with a S/N:             457,425  (20.1%)
  with no DR17 match:                               1,815,450

Where the S/N we use comes from:
  Astra S/N available:                              1,630,104
  Astra S/N missing, DR17 S/N available (fallback):   457,388
  no S/N from either source:                          185,383


## 2. Quality cuts (before the chemistry)

These cuts remove stars with unreliable spectra, parameters or distances, and select red giants. They are applied one after another, and the count after each is printed and saved to the log. We do not cut on radial velocity or proper motion.

| # | Cut | What it does |
|---|-----|--------------|
| 1 | Finite values | Teff, logg, [Fe/H], [Mg/H] and S/N must all be finite. |
| 2 | Giant branch | (Teff, logg) must lie inside the red giant branch polygon, with Teff 3000 to 7500 K and logg between -2 and 6.5. |
| 3 | S/N | Spectral S/N > 20, taken from the Astra `snr` column where available and from DR17 otherwise. |
| 4 | `flag_bad` | The ASPCAP `flag_bad` flag must not be set. In Astra it is built from the ASPCAP `result_flags` and is set if the fit shows any sign of failure or unreliability (suspicious or unphysical parameters, high reduced chi-squared, high v sin i or microturbulence, a parameter at the model grid edge, a failed FERRE fit, low S/N, or a spectrum read error). It is the closest analogue of the DR17 `STAR_BAD` flag. |
| 5 | Abundance flags | `fe_h_flags` = 0 and `mg_h_flags` = 0: no censoring, grid-edge or Teff warning flags on [Fe/H] or [Mg/H]. |
| 6 | Cluster members | `sdss4_apogee_member_flags` = 0: no "likely member of a globular or open cluster" bit is set. These flags come from the SDSS-IV APOGEE reduction. |
| 7 | Parallax | ZGR parallax (`zgr_plx`) must be finite and positive. Distance is 1 / parallax. There is no parallax S/N cut. |
| 8 | ZGR quality | `zgr_quality_flags` < 8 (Zhang, Green & Rix 2023, the recommended basic reliability cut). This removes poor SED fits, stars unlikely under the model prior, and stars whose ZGR parallax disagrees with Gaia DR3 by more than 10 sigma. |
| 9 | Sky position | `ra` and `dec` must be finite, since they are needed to compute a position. |

In [5]:
ZGR_QUALITY_MAX = 8      # keep zgr_quality_flags < 8

def apply_cut(table, mask, label):
    """Keep the rows where mask is True, record the count, and print one line."""
    out = table[np.asarray(mask, bool)]
    cut_log.append((label, len(out)))
    print(f"{label:<56s} {len(out):>10,d}  ({len(out) - len(table):+,d})")
    return out

print(f"{'cut':<56s} {'stars left':>10s}  (removed)")
print(f"{'start: parent catalog':<56s} {len(t):>10,d}")

t_sel = t

# 1. finite values -----------------------------------------------------------
teff = col_to_float_array(t_sel, "teff")
logg = col_to_float_array(t_sel, "logg")
feh  = col_to_float_array(t_sel, "fe_h")
mgh  = col_to_float_array(t_sel, "mg_h")
snr  = masked_fallback_array(t_sel, "snr", "SNR")
t_sel = apply_cut(t_sel,
    np.isfinite(teff) & np.isfinite(logg) & np.isfinite(feh) & np.isfinite(mgh) & np.isfinite(snr),
    "1. finite Teff, logg, [Fe/H], [Mg/H], S/N")

# 2. giant branch ------------------------------------------------------------
teff = col_to_float_array(t_sel, "teff")
logg = col_to_float_array(t_sel, "logg")
t_sel = apply_cut(t_sel, giant_branch_mask(teff, logg, build_gb_polygon()),
    "2. giant branch polygon")

# 3. signal to noise ---------------------------------------------------------
snr = masked_fallback_array(t_sel, "snr", "SNR")
t_sel = apply_cut(t_sel, snr > SNR_MIN, f"3. S/N > {SNR_MIN}")

# 4. flag_bad ----------------------------------------------------------------
t_sel = apply_cut(t_sel, ~np.asarray(t_sel["flag_bad"], bool), "4. flag_bad not set")

# 5. abundance flags ---------------------------------------------------------
t_sel = apply_cut(t_sel,
    (np.asarray(t_sel["fe_h_flags"]) == 0) & (np.asarray(t_sel["mg_h_flags"]) == 0),
    "5. fe_h_flags == 0 and mg_h_flags == 0")

# 6. cluster members ---------------------------------------------------------
t_sel = apply_cut(t_sel,
    np.asarray(t_sel["sdss4_apogee_member_flags"], dtype=np.int64) == 0,
    "6. not a flagged cluster member")

# 7. parallax ----------------------------------------------------------------
plx = col_to_float_array(t_sel, "zgr_plx")
t_sel = apply_cut(t_sel, np.isfinite(plx) & (plx > 0), "7. zgr_plx finite and > 0")

# 8. ZGR quality -------------------------------------------------------------
t_sel = apply_cut(t_sel,
    np.asarray(t_sel["zgr_quality_flags"], dtype=np.int64) < ZGR_QUALITY_MAX,
    f"8. zgr_quality_flags < {ZGR_QUALITY_MAX}")

# 9. sky position ------------------------------------------------------------
t_sel = apply_cut(t_sel,
    np.isfinite(col_to_float_array(t_sel, "ra")) & np.isfinite(col_to_float_array(t_sel, "dec")),
    "9. finite ra, dec")

print(f"\nstars after the quality cuts: {len(t_sel):,}")

cut                                                      stars left  (removed)
start: parent catalog                                     2,272,875
1. finite Teff, logg, [Fe/H], [Mg/H], S/N                 2,035,097  (-237,778)
2. giant branch polygon                                   1,397,052  (-638,045)
3. S/N > 20                                               1,385,793  (-11,259)
4. flag_bad not set                                       1,351,059  (-34,734)
5. fe_h_flags == 0 and mg_h_flags == 0                    1,348,238  (-2,821)
6. not a flagged cluster member                           1,345,438  (-2,800)
7. zgr_plx finite and > 0                                 1,154,223  (-191,215)
8. zgr_quality_flags < 8                                    550,159  (-604,064)
9. finite ra, dec                                           550,159  (+0)

stars after the quality cuts: 550,159


## 3. Chemistry cuts

**3a. Low-alpha disk.** We keep stars whose [Mg/Fe] is below a two-piece straight line in ([Fe/H], [Mg/Fe]). For [Fe/H] >= -0.1 the line is [Mg/Fe] = -0.05 [Fe/H] + 0.10, and for [Fe/H] < -0.1 it is [Mg/Fe] = -0.20 [Fe/H] + 0.085. The two pieces meet at [Fe/H] = -0.1. Stars with non-finite [Fe/H] or [Mg/Fe] are dropped.

**3b. Metallicity floor from the analysis code.** [Fe/H] >= -1.2.

**3c. Extra metallicity floor.** [Fe/H] > -0.9. This is tighter than 3b, so it is the effective floor of the sample.

In [6]:
from midplane.chemistry import apply_low_alpha_cut, FEH_CUT     # FEH_CUT = -1.2

print(f"{'cut':<56s} {'stars left':>10s}  (removed)")
print(f"{'start: after the quality cuts':<56s} {len(t_sel):>10,d}")

# 3a. low-alpha disk (feh_cut = -inf so only the dividing line is applied here)
low = apply_low_alpha_cut(t_sel, feh_cut=-np.inf)
n_before = len(t_sel)
t_sel = low["table"]
fe = np.asarray(low["fe"])                  # [Fe/H] of the stars we kept, same order as t_sel
cut_log.append(("10. low-alpha ([Mg/Fe] below the dividing line)", len(t_sel)))
print(f"{'10. low-alpha ([Mg/Fe] below the dividing line)':<56s} {len(t_sel):>10,d}  ({len(t_sel) - n_before:+,d})")

# 3b. metallicity floor from the analysis code
keep = fe >= FEH_CUT
t_sel = apply_cut(t_sel, keep, f"11. [Fe/H] >= {FEH_CUT}")
fe = fe[keep]

# 3c. extra metallicity floor (the effective one)
keep = fe > FEH_MIN_EXTRA
t_sel = apply_cut(t_sel, keep, f"12. [Fe/H] > {FEH_MIN_EXTRA}")
fe = fe[keep]

print(f"\nstars after the chemistry cuts: {len(t_sel):,}")
print(f"[Fe/H] range of the sample: {fe.min():.2f} to {fe.max():.2f}")

cut                                                      stars left  (removed)
start: after the quality cuts                               550,159
10. low-alpha ([Mg/Fe] below the dividing line)             421,010  (-129,149)
11. [Fe/H] >= -1.2                                          413,120  (-7,890)
12. [Fe/H] > -0.9                                           406,589  (-6,531)

stars after the chemistry cuts: 406,589
[Fe/H] range of the sample: -0.90 to 0.76


## 4. [Al/Fe] vs [Mg/Mn] polygon

We keep stars that lie inside a four-sided polygon in the ([Al/Fe], [Mg/Mn]) plane, where [Al/Fe] = [Al/H] - [Fe/H] and [Mg/Mn] = [Mg/H] - [Mn/H]. Stars with a non-finite value of either quantity are dropped.

Polygon vertices ([Al/Fe], [Mg/Mn]): (-0.4, -0.25), (0.15, -0.40), (0.6, 0.7), (-0.2, 0.7).

In [7]:
from midplane.chemistry import apply_alfe_mgmn_polygon_cut

n_before = len(t_sel)
t_final, poly_diag, poly_arrays = apply_alfe_mgmn_polygon_cut(t_sel)

cut_log.append(("13. inside the [Al/Fe]-[Mg/Mn] polygon", len(t_final)))
print(f"{'13. inside the [Al/Fe]-[Mg/Mn] polygon':<56s} {len(t_final):>10,d}  ({len(t_final) - n_before:+,d})")
print(f"\nstars with finite [Al/Fe] and [Mg/Mn]: {poly_diag['finite_points']:,} of {poly_diag['input_size']:,}")
print(f"fraction of those inside the polygon:  {poly_diag['inside_fraction_finite']:.3f}")
print(f"stars after the polygon cut: {len(t_final):,}")

13. inside the [Al/Fe]-[Mg/Mn] polygon                      347,563  (-59,026)

stars with finite [Al/Fe] and [Mg/Mn]: 391,662 of 406,589
fraction of those inside the polygon:  0.887
stars after the polygon cut: 347,563


## 5. Computed columns (no stars removed)

We add the following columns to the table for every star. Nothing is cut here, including no cut on radius.

| Column | Meaning |
|---|---|
| `dist_kpc` | Distance in kpc, 1 / ZGR parallax |
| `x_gc_kpc`, `y_gc_kpc`, `z_gc_kpc` | Galactocentric Cartesian position in kpc |
| `R_gc_kpc` | Cylindrical radius, sqrt(x² + y²) |
| `phi_gc_rad` | Azimuth, arctan2(y, x), in radians |
| `mgfe` | [Mg/Fe] = [Mg/H] - [Fe/H] |
| `mgfe_err` | Uncertainty on [Mg/Fe], sqrt(e_mg_h² + e_fe_h²). It is NaN where either error is missing. |
| `alfe`, `mgmn` | [Al/Fe] and [Mg/Mn], the quantities used in the polygon cut |
| `snr_used` | The S/N used in the S/N cut |
| `snr_source` | 0 if `snr_used` is the Astra `snr`, 1 if it is the DR17 `SNR` fallback |

The Galactocentric frame has the Sun at 8.275 kpc from the Galactic centre and exactly on the midplane (z_sun = 0). Only positions are computed, so proper motions and radial velocities are not needed.

In [8]:
GALCEN_DISTANCE_KPC = 8.275
GALCEN_V_SUN = (8, 254, 8)        # km/s (only defines the frame; no velocities are used)

new_cols = ["dist_kpc", "x_gc_kpc", "y_gc_kpc", "z_gc_kpc", "R_gc_kpc", "phi_gc_rad",
            "mgfe", "mgfe_err", "alfe", "mgmn", "snr_used", "snr_source"]
clash = [c for c in new_cols if c in t_final.colnames]
assert not clash, f"these column names already exist: {clash}"

# --- positions ---
plx = col_to_float_array(t_final, "zgr_plx")                # mas
dist_kpc = 1.0 / plx                                         # kpc
c = SkyCoord(ra=col_to_float_array(t_final, "ra") * u.deg,
             dec=col_to_float_array(t_final, "dec") * u.deg,
             distance=Distance(dist_kpc * u.kpc))
frame = coord.Galactocentric(galcen_distance=GALCEN_DISTANCE_KPC * u.kpc,
                             galcen_v_sun=list(GALCEN_V_SUN) * u.km / u.s,
                             z_sun=0 * u.pc)
gc = c.transform_to(frame)
x, y, z = (gc.x.to_value(u.kpc), gc.y.to_value(u.kpc), gc.z.to_value(u.kpc))

t_final["dist_kpc"]   = dist_kpc
t_final["x_gc_kpc"]   = x
t_final["y_gc_kpc"]   = y
t_final["z_gc_kpc"]   = z
t_final["R_gc_kpc"]   = np.sqrt(x**2 + y**2)
t_final["phi_gc_rad"] = np.arctan2(y, x)

# --- chemistry ---
feh = col_to_float_array(t_final, "fe_h")
mgh = col_to_float_array(t_final, "mg_h")
t_final["mgfe"]     = mgh - feh
t_final["mgfe_err"] = np.sqrt(col_to_float_array(t_final, "e_mg_h")**2
                              + col_to_float_array(t_final, "e_fe_h")**2)
t_final["alfe"] = col_to_float_array(t_final, "al_h") - feh
t_final["mgmn"] = mgh - col_to_float_array(t_final, "mn_h")

# --- S/N actually used, and where it came from ---
snr_astra = col_to_float_array(t_final, "snr")
t_final["snr_used"]   = masked_fallback_array(t_final, "snr", "SNR")
t_final["snr_source"] = np.where(np.isfinite(snr_astra), 0, 1).astype(np.int8)

# --- units ---
for name in ["dist_kpc", "x_gc_kpc", "y_gc_kpc", "z_gc_kpc", "R_gc_kpc"]:
    t_final[name].unit = u.kpc
t_final["phi_gc_rad"].unit = u.rad

# --- summary (nothing is cut here) ---
print(f"stars: {len(t_final):,}   (unchanged)")
for name in ["dist_kpc", "R_gc_kpc", "z_gc_kpc", "mgfe", "mgfe_err"]:
    v = np.asarray(t_final[name], float)
    print(f"{name:<10s} finite: {np.isfinite(v).sum():>8,d}   "
          f"min {np.nanmin(v):8.3f}   median {np.nanmedian(v):8.3f}   max {np.nanmax(v):8.3f}")
print(f"S/N from Astra: {(t_final['snr_source'] == 0).sum():,}   from DR17 fallback: {(t_final['snr_source'] == 1).sum():,}")

stars: 347,563   (unchanged)
dist_kpc   finite:  347,563   min    0.075   median    2.119   max   36.916
R_gc_kpc   finite:  347,563   min    0.058   median    9.042   max   33.385
z_gc_kpc   finite:  347,563   min  -20.905   median   -0.008   max   12.926
mgfe       finite:  347,563   min   -0.510   median    0.010   max    0.260
mgfe_err   finite:  338,113   min    0.002   median    0.005   max    0.051
S/N from Astra: 220,927   from DR17 fallback: 126,636


## 6. Save the sample

Two files are written to `data/`:

- `final_sample_zsun0.fits`: every original catalog column for the stars that passed all the cuts, plus the computed columns from the previous section.
- `final_sample_cut_log.csv`: the number of stars left after each cut.

In [12]:
# --- 1. Cut log: stars remaining after each step -----------------------------
log_df = pd.DataFrame(cut_log, columns=["step", "stars_remaining"])
log_df["stars_removed"] = (-log_df["stars_remaining"].diff()).astype("Int64")
                                       # negative = rows added (the DR17 join), blank = starting point
log_df.to_csv(OUT_LOG, index=False)
print(log_df.to_string(index=False))

# --- 2. Record the cut parameters in the FITS header --------------------------
t_final.meta.update({
    "GCDIST":  GALCEN_DISTANCE_KPC,   # Sun-Galactic centre distance [kpc]
    "ZSUNPC":  0,                     # Sun height above the midplane [pc]
    "SNRMIN":  SNR_MIN,               # spectral S/N cut
    "ZGRQMAX": ZGR_QUALITY_MAX,       # keep zgr_quality_flags below this
    "FEHMIN":  FEH_MIN_EXTRA,         # effective [Fe/H] floor
})

# --- 3. Write the sample ------------------------------------------------------
t_final.write(OUT_SAMPLE, format="fits", overwrite=True)

# --- 4. Reload and check ------------------------------------------------------
check = Table.read(OUT_SAMPLE, format="fits")
assert len(check) == len(t_final), "row count differs after reloading"
assert check.colnames == t_final.colnames, "columns differ after reloading"
assert np.allclose(check["R_gc_kpc"], t_final["R_gc_kpc"], equal_nan=True)

print(f"\nSaved {len(check):,} stars and {len(check.colnames)} columns "
      f"({OUT_SAMPLE.stat().st_size / 1e6:,.0f} MB)")
print(f"  sample:  {OUT_SAMPLE}")
print(f"  cut log: {OUT_LOG}")

                                            step  stars_remaining  stars_removed
               0a. Astra ASPCAP rows in the file          2466006           <NA>
0b. after joining DR17 (includes duplicate rows)          2576187        -110181
        0c. one row per sdss_id (parent catalog)          2272875         303312
       1. finite Teff, logg, [Fe/H], [Mg/H], S/N          2035097         237778
                         2. giant branch polygon          1397052         638045
                                     3. S/N > 20          1385793          11259
                             4. flag_bad not set          1351059          34734
          5. fe_h_flags == 0 and mg_h_flags == 0          1348238           2821
                 6. not a flagged cluster member          1345438           2800
                       7. zgr_plx finite and > 0          1154223         191215
                        8. zgr_quality_flags < 8           550159         604064
                            